In [ ]:
# standard libraries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from datetime import datetime
dateparse = lambda x: datetime.strptime(x, '%Y:%m:%d')
import re, string
from string import digits

# transformers
from transformers import AutoModelForSequenceClassification
from transformers import TFAutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig
from scipy.special import softmax

# visualisation
import seaborn as sns
from matplotlib import pyplot
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

### Load Data

In [ ]:
# load data
ak = pd.read_csv('../input/201k-tweets-on-mrmodimrrahulmrkejrielecanal/Arvind Kejriwal_data.csv', parse_dates=['Date'], date_parser=dateparse)
nm = pd.read_csv('../input/201k-tweets-on-mrmodimrrahulmrkejrielecanal/Narendra Modi_data.csv', parse_dates=['Date'], date_parser=dateparse)
rg = pd.read_csv('../input/201k-tweets-on-mrmodimrrahulmrkejrielecanal/Rahul Gandhi_data.csv', parse_dates=['Date'], date_parser=dateparse)

# sample 'n' records from each datasets [due to memory restrictions]
n = 1000
rand_int = np.random.randint(100)
ak = ak.sample(n, replace=True, random_state=rand_int)
nm = nm.sample(n, replace=True, random_state=rand_int)
rg = rg.sample(n, replace=True, random_state=rand_int)

# adding an additional column ('about') for classification
'''ak : arvind kejriwal, nm : narendra modi, rg : rahul gandhi'''
ak['about'] = 'ak'
nm['about'] = 'nm'
rg['about'] = 'rg'

# merge all dataframes
df = pd.concat([ak, nm, rg], axis=0)

# drop NULLs (if any)
df.dropna(inplace=True)

# drop 'Time' column
df.drop('Time',axis=1, inplace=True)

# reset index
df.reset_index(drop=True, inplace=True)

# view
print(f'Sampled {df.shape[0]} tweets in total')

### Data Cleanup & Prep

In [ ]:
# custom function to remove @ mentions, URLs and special characters
''' note that this function will also remove non-english chars as well if there are any'''
def clean(txt):
  #removing url & @ mentions
  proc_txt = re.sub(r"(?:\@|http?\://|https?\://|www)\S+", "", txt)
  proc_txt = " ".join(proc_txt.split())

  #removing special chars
  proc_txt2 = re.sub(r"[^A-Za-z0-9]+"," ",proc_txt)
  proc_txt2 = " ".join(proc_txt2.split())
  
  # remove punctuations (residual if any)
  punct_en = set(string.punctuation)
  proc_txt2 = ''.join(char1 for char1 in proc_txt2 if char1 not in punct_en)
    
  # remove numericals (residual if any)
  num_digits = str.maketrans('','', digits)
  proc_txt2 = proc_txt2.translate(num_digits)

  # return the processed txt in lower case and without white spaces (if there are any)
  return proc_txt2.lower().strip()

In [ ]:
# applying the function to 'Tweets' column to create a new 'cleaned' column
df['cleaned_Tweet'] = df['Tweet'].apply(lambda x: clean(x))

# view
df.head()

### Sentiment Analysis

**Will be using HuggingFace's pretrained (on Twitter data) Model for sentiment analysis. For more details about the model, please refer** [here](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest?text=Covid+cases+are+increasing+fast%21)

In [ ]:
# define the model name
model_name = f"cardiffnlp/twitter-roberta-base-sentiment-latest"

# define the tokenizer for the model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# define the configs for the model
config = AutoConfig.from_pretrained(model_name)

# get the pretrained model
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
# custom function to get the sentiments
def get_sentiments(tweet):
    encoded_input = tokenizer(tweet, return_tensors='pt')
    output = model(**encoded_input)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    ranking = np.argsort(scores)
    ranking = ranking[::-1]
    label = config.id2label[ranking[0]]
    return label

In [ ]:
# apply the function to the 'cleaned tweet'
df['sentiments'] = df['cleaned_Tweet'].apply(lambda x: get_sentiments(x))

# view
df.head()

### Visualisation

In [ ]:
# visualising Political Sentiments Distribution
temp = df.groupby(['about','sentiments']).size().reset_index().rename(columns={0:'count'})
x_axis = temp['sentiments'].unique()

# create a initials-name dictionary
name_dict = {'ak':'Arvind.K', 'nm':'Narendra.M', 'rg':'Rahul.G'}
# create a initials-hex color dictionary
hex_col = {'ak':'khaki', 'nm':'darkslategrey', 'rg':'lightcoral'}

# instantiate figure
fig = go.Figure()

# add traces
for t in temp['about'].unique():
    fig.add_trace(go.Bar(
                    x=x_axis,
                    y=temp[temp['about'] == t]['count'].tolist(),
                    name=name_dict[t],
                    marker_color=hex_col[t]
                ))
fig.update_traces(texttemplate='%{y:.2s}', textposition='inside')
fig.update_layout(barmode='relative', xaxis_tickangle=0)
fig.update_layout(title= 'Political Sentiments Distribution',
                  title_x=0.5,
                  titlefont_size=20,
                  )
fig.show()

### Negative Sentiments - Visualisation

In [ ]:
# Visualising negative sentiments 
temp1 = df.groupby(['about','sentiments','Date']).size().reset_index().rename(columns={0:'count'})

# create a initials-name dictionary
name_dict = {'ak':'Arvind.K', 'nm':'Narendra.M', 'rg':'Rahul.G'}
# create a initials-hex color dictionary
hex_col = {'ak':'#297373', 'nm':'#FF8552', 'rg':'#39393A'}

# instantiate figure
fig = go.Figure()

# add traces
for t in temp1['about'].unique():
    fig.add_trace(go.Scatter(
                    x=temp1[(temp1['sentiments'] == 'Negative') & (temp1['about'] == t)]['Date'].tolist(),
                    y=temp1[(temp1['sentiments'] == 'Negative') & (temp1['about'] == t)]['count'].tolist(),
                    name=name_dict[t],
                    mode='markers',
                    marker=dict(
                                size=10,
                                color= hex_col[t],
                                showscale=False
                    
                    )
                ))
fig.update_layout(barmode='relative', xaxis_tickangle=0)
fig.update_layout(title= 'Negative Sentiments Distribution Overtime',
                  title_x=0.5,
                  titlefont_size=20,
                  )
fig.show()

### Postive Sentiments - Visualisation

In [ ]:
# Visualising negative sentiments 
temp2 = df.groupby(['about','sentiments','Date']).size().reset_index().rename(columns={0:'count'})

# create a initials-name dictionary
name_dict = {'ak':'Arvind.K', 'nm':'Narendra.M', 'rg':'Rahul.G'}
# create a initials-hex color dictionary
hex_col = {'ak':'#297373', 'nm':'#FF8552', 'rg':'#39393A'}

# instantiate figure
fig = go.Figure()

# add traces
for t in temp1['about'].unique():
    fig.add_trace(go.Scatter(
                    x=temp2[(temp2['sentiments'] == 'Positive') & (temp2['about'] == t)]['Date'].tolist(),
                    y=temp2[(temp2['sentiments'] == 'Positive') & (temp2['about'] == t)]['count'].tolist(),
                    name=name_dict[t],
                    mode='markers',
                    marker=dict(
                                size=10,
                                color= hex_col[t],
                                showscale=False
                    
                    )
                ))
fig.update_layout(barmode='relative', xaxis_tickangle=0)
fig.update_layout(title= 'Positive Sentiments Distribution Overtime',
                  title_x=0.5,
                  titlefont_size=20,
                  )
fig.show()